# Tiny ImageNet frozen representation evaluation

Purpose: cache complete-image EMA features, fit a labeled linear probe without changing encoders, and measure image-level dispersion before standardization. Inputs: pinned split memberships and initialized/trained teacher states. Outputs: CPU feature caches, epoch probe checkpoints, predictions, metrics and conditional paired bootstrap intervals. Expected runtime: seconds for unit fixtures; minutes for smoke; pilot depends on device. Extraction resumes from `<cache key>.progress.pt`; rerun `extract_features` with identical arguments. Probe resumes at epoch boundaries by rerunning `fit_linear_probe` with the same configuration and output directory. A corrupt or mismatched cache/checkpoint raises. No dataset download or training executes on import.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


## Feature and uncertainty contract

The imported encoder applies final LayerNorm to each patch. Evaluation calls the full encoder, then takes the arithmetic mean of all valid final patch vectors. The predictor is absent. Feature standardization uses only the probe-training pool and is not used in collapse diagnostics. The classifier is exactly `Linear(D, 200)` with cross-entropy. Development selects the best epoch with earliest ties from the same fixed number of epochs for all four representations. Held-out is unavailable in smoke mode.

For centered image embeddings, the covariance effective rank is `exp(-sum(q*log(q)))`, where `q` are normalized nonnegative covariance eigenvalues. Exactly zero variance gives rank zero. The rank ceiling is `min(feature_dim, images - 1)`. Registered `trace <= 1e-12` and effective rank `< 2` flags are advisory diagnostics, not universal theorems. Matching-position token dispersion is measured across images, so within-image position signals cannot alone satisfy it.

Paired bootstrap resamples evaluation images together for two fixed representations. Its intervals exclude uncertainty from training seeds.


In [ ]:
"""Frozen EMA image features, linear probes, collapse diagnostics and paired comparisons.
No training labels enter JEPA. Only notebook ahr-export definition cells are imported.
"""
from contextlib import contextmanager, nullcontext
from pathlib import Path
import copy, csv, gc, hashlib, io, json, math, os, random, statistics, time, uuid
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from fmm_jepa.training import atomic_json, digest_file
from fmm_jepa.geometry import clear_geometry_cache

FEATURE_SCHEMA = 'fmm-jepa-tiny-features/v1'
PROBE_SCHEMA = 'fmm-jepa-tiny-probe/v1'
REQUIRED_IDENTITY = ('dataset_revision', 'sample_ids', 'checkpoint_hash', 'encoder_kind',
                     'preprocessing', 'pooling', 'normalization')

def stable_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()

def encoder_state_hash(encoder):
    state = encoder.state_dict() if hasattr(encoder, 'state_dict') else encoder
    digest = hashlib.sha256()
    for key in sorted(state):
        tensor = state[key].detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape)).encode())
        digest.update(tensor.reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()

def atomic_torch(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        with temp.open('wb') as handle:
            torch.save(value, handle); handle.flush(); os.fsync(handle.fileno())
        os.replace(temp, path)
        descriptor = os.open(path.parent, os.O_DIRECTORY)
        try: os.fsync(descriptor)
        finally: os.close(descriptor)
    finally:
        if temp.exists(): temp.unlink()

def write_csv(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row))
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        with temp.open('w', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=fields); writer.writeheader()
            for row in rows:
                writer.writerow({k: json.dumps(v) if isinstance(v, (dict, list)) else v for k,v in row.items()})
            handle.flush(); os.fsync(handle.fileno())
        os.replace(temp, path)
    finally:
        if temp.exists(): temp.unlink()

@contextmanager
def preserve_randomness():
    python = random.getstate(); numpy = np.random.get_state(); cpu = torch.get_rng_state()
    cuda = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try: yield
    finally:
        random.setstate(python); np.random.set_state(numpy); torch.set_rng_state(cpu)
        if cuda is not None: torch.cuda.set_rng_state_all(cuda)

@contextmanager
def frozen_encoder(encoder, device=None):
    # Geometry constants created inside inference_mode are inference tensors.
    # Never let that evaluation-only cache escape into later autograd training.
    clear_geometry_cache()
    modes = [(module, module.training) for module in encoder.modules()]
    flags = [(p, p.requires_grad) for p in encoder.parameters()]
    original_device = next(encoder.parameters()).device
    before = encoder_state_hash(encoder)
    try:
        encoder.eval().requires_grad_(False)
        if device is not None: encoder.to(device)
        with preserve_randomness():
            yield encoder
        if encoder_state_hash(encoder) != before:
            raise RuntimeError('Frozen encoder state changed during evaluation')
    finally:
        clear_geometry_cache()
        encoder.to(original_device)
        for param, flag in flags: param.requires_grad_(flag)
        for module, mode in modes: module.training = mode

class EvaluationDataset:
    """Adapt a pinned TinyDataset + ordered membership IDs to deterministic images."""
    def __init__(self, dataset, sample_ids, transform):
        self.dataset, self.sample_ids, self.transform = dataset, list(sample_ids), transform
    def __len__(self): return len(self.sample_ids)
    def __getitem__(self, index):
        sample_id = self.sample_ids[index]
        record = self.dataset.get(sample_id)
        image = record['image']
        return {'image': self.transform(image), 'label': int(record['label']), 'sample_id': sample_id}

def _batch(dataset, start, stop):
    items = [dataset[i] for i in range(start, stop)]
    images = torch.stack([item['image'] for item in items])
    labels = torch.tensor([int(item['label']) for item in items], dtype=torch.long)
    ids = [str(item['sample_id']) for item in items]
    return images, labels, ids

def _sync(device):
    if torch.device(device).type == 'cuda': torch.cuda.synchronize(device)

def _memory_start(device):
    if torch.device(device).type != 'cuda': return None
    _sync(device); torch.cuda.reset_peak_memory_stats(device)
    return int(torch.cuda.memory_allocated(device))

def _performance(phase, seconds, count, times, device, baseline, **metadata):
    cuda = torch.device(device).type == 'cuda'
    median = statistics.median(times) if times else None
    return {'phase': phase, 'status': 'PASS', 'device': str(device), 'precision':'float32',
            'wall_seconds':seconds, 'examples':count, 'examples_per_second':count / seconds if seconds > 0 else None,
            'median_batch_seconds':median,
            'mad_batch_seconds':statistics.median(abs(x-median) for x in times) if times else None,
            'batches_measured':len(times), 'warmup_batches_excluded':0,
            'baseline_allocated_bytes':baseline,
            'peak_allocated_bytes':int(torch.cuda.max_memory_allocated(device)) if cuda else None,
            'memory_status':'measured_cuda_allocator' if cuda else 'NOT_RUN_CPU_tensor_peak',
            'memory_definition':'CUDA allocation peak includes live model and inputs; excludes reserved cache. CPU peak unmeasured.',
            **metadata}

def validate_feature_cache(cache, identity):
    if cache.get('schema') != FEATURE_SCHEMA or cache.get('identity') != identity:
        raise RuntimeError('Stale feature cache: schema or identity differs; never silently reuse')
    features, labels, ids = cache['features'], cache['labels'], cache['sample_ids']
    if features.ndim != 2 or labels.ndim != 1 or len(ids) != len(features) or len(labels) != len(ids):
        raise RuntimeError('Invalid feature cache dimensions')
    if ids != identity['sample_ids'] or len(set(ids)) != len(ids):
        raise RuntimeError('Feature sample IDs/order do not match pinned membership')
    if features.device.type != 'cpu' or labels.device.type != 'cpu' or not torch.isfinite(features).all():
        raise RuntimeError('Features must be finite CPU tensors')
    if labels.dtype != torch.long or bool(((labels < 0) | (labels >= 200)).any()):
        raise RuntimeError('Invalid Tiny ImageNet labels')
    if cache.get('content_hash') != encoder_state_hash({'features': features, 'labels': labels}):
        raise RuntimeError('Feature cache content hash mismatch')
    return cache

def extract_features(encoder, dataset, cache_dir, identity, batch_size=32, device=None):
    """Complete-image EMA output is final LayerNorm -> mean over all 64 valid patches.
    CPU caches are keyed by full provenance. Progress saves every 32 batches or 15 min.
    A complete cache is committed atomically; corrupted/stale entries raise.
    """
    if any(field not in identity for field in REQUIRED_IDENTITY): raise ValueError('Incomplete feature cache identity')
    if len(dataset) != len(identity['sample_ids']) or not len(dataset): raise ValueError('Dataset membership mismatch/empty')
    if identity['pooling'] != 'mean_final_layernorm_patches': raise ValueError('Unregistered pooling')
    if identity['checkpoint_hash'] != encoder_state_hash(encoder): raise ValueError('Encoder state hash differs from checkpoint identity')
    if identity['encoder_kind'] != getattr(encoder, 'kind', identity['encoder_kind']): raise ValueError('Encoder kind mismatch')
    if batch_size < 1: raise ValueError('Positive feature batch size required')
    key = stable_digest(identity); folder = Path(cache_dir); folder.mkdir(parents=True, exist_ok=True)
    path = folder / (key + '.pt'); manifest = folder / (key + '.json'); progress_path = folder / (key + '.progress.pt')
    if path.exists() or manifest.exists():
        if not path.exists() or not manifest.exists(): raise RuntimeError('Incomplete committed feature cache; inspect before recovery')
        meta = json.loads(manifest.read_text())
        if meta.get('sha256') != digest_file(path): raise RuntimeError('Feature cache file hash mismatch')
        cache = validate_feature_cache(torch.load(path, map_location='cpu', weights_only=True), identity)
        return {**cache, 'path': str(path), 'reused': True,
                'performance': {'phase':'feature_extraction','status':'NOT_RUN','reason':'valid cache reused','cache_key':key}}
    feature_parts=[]; label_parts=[]; ids=[]; start_at=0
    if progress_path.exists():
        progress = torch.load(progress_path, map_location='cpu', weights_only=True)
        if progress.get('identity') != identity or progress.get('schema') != FEATURE_SCHEMA:
            raise RuntimeError('Invalid feature extraction resume identity')
        feature_parts=progress['feature_parts']; label_parts=progress['label_parts']; ids=progress['sample_ids']
        start_at=len(ids)
        if ids != identity['sample_ids'][:start_at] or sum(len(x) for x in feature_parts) != start_at:
            raise RuntimeError('Invalid extraction resume position')
    device = torch.device(device or next(encoder.parameters()).device)
    times=[]; last_save=time.monotonic(); extracted=0
    with frozen_encoder(encoder, device), torch.inference_mode():
        baseline = _memory_start(device); _sync(device); started=time.perf_counter()
        for index, start in enumerate(range(start_at, len(dataset), batch_size)):
            _sync(device); tick=time.perf_counter()
            images, labels, batch_ids = _batch(dataset, start, min(start+batch_size, len(dataset)))
            if batch_ids != identity['sample_ids'][start:start+len(batch_ids)]: raise RuntimeError('Dataset order differs from cache identity')
            tokens=encoder(images.to(device, dtype=torch.float32))
            if tokens.ndim != 3 or tokens.shape[0] != len(batch_ids): raise RuntimeError('Unexpected encoder output')
            if not torch.isfinite(tokens).all(): raise FloatingPointError('Nonfinite encoder features')
            feature_parts.append(tokens.mean(dim=1).float().cpu().clone()); label_parts.append(labels); ids.extend(batch_ids)
            extracted += len(batch_ids)
            del images,tokens; _sync(device); times.append(time.perf_counter()-tick)
            if (index+1) % 32 == 0 or time.monotonic()-last_save >= 900:
                atomic_torch(progress_path, {'schema':FEATURE_SCHEMA,'identity':identity,'feature_parts':feature_parts,
                                            'label_parts':label_parts,'sample_ids':ids,'resume_next_index':len(ids)})
                last_save=time.monotonic()
        _sync(device); seconds=time.perf_counter()-started
        performance=_performance('feature_extraction',seconds,extracted,times,device,baseline,
                                 includes_data_loading=True,includes_preprocessing=True,includes_diagnostics=False,
                                 includes_optimizer=False,N_grid=64,N_visible=64,batch_size=batch_size,cache_key=key,
                                 resumed_examples=start_at,includes_progress_checkpoint_writes=True)
    features=torch.cat(feature_parts); labels=torch.cat(label_parts)
    cache={'schema':FEATURE_SCHEMA,'identity':copy.deepcopy(identity),'features':features,'labels':labels,'sample_ids':ids,
           'content_hash':encoder_state_hash({'features':features,'labels':labels})}
    validate_feature_cache(cache, identity); atomic_torch(path, cache)
    atomic_json(manifest, {'schema':FEATURE_SCHEMA,'cache_key':key,'sha256':digest_file(path),'identity':identity,
                           'examples':len(ids),'shape':list(features.shape),'performance':performance})
    if progress_path.exists(): progress_path.unlink()
    return {**cache,'path':str(path),'reused':False,'performance':performance}

def fit_standardization(features, enabled=True):
    if features.ndim != 2 or len(features)<1: raise ValueError('Nonempty 2-D training features required')
    if not enabled: return {'mean':torch.zeros(features.shape[1]),'std':torch.ones(features.shape[1]),'enabled':False,'fit_examples':len(features)}
    return {'mean':features.float().mean(0),'std':features.float().std(0,unbiased=False).clamp_min(1e-6),
            'enabled':True,'fit_examples':len(features),'scope':'probe_training_features_only','epsilon':1e-6}

def standardize(features, stats): return (features.float()-stats['mean']) / stats['std']

def classification_metrics(logits, labels):
    if logits.ndim != 2 or logits.shape[1] != 200 or len(logits) != len(labels): raise ValueError('Expected [n,200] logits and n labels')
    if not len(labels): raise ValueError('Empty evaluation')
    if not torch.isfinite(logits).all(): raise FloatingPointError('Nonfinite probe logits')
    top=logits.topk(5,dim=1).indices
    return {'loss':float(F.cross_entropy(logits,labels)),'top1':float((top[:,0]==labels).float().mean()),
            'top5':float((top==labels[:,None]).any(1).float().mean()),'examples':len(labels),
            'evaluated_classes':int(labels.unique().numel())}

def prediction_rows(logits, labels, sample_ids, model_id, split, checkpoint_hash, probe_hash, seed):
    if len(sample_ids)!=len(labels) or len(set(sample_ids))!=len(sample_ids): raise ValueError('Invalid prediction IDs')
    top=logits.topk(5,dim=1).indices.cpu()
    return [{'model_id':model_id,'seed':seed,'phase':'linear_probe','split':split,'checkpoint_hash':checkpoint_hash,
             'probe_hash':probe_hash,'sample_id':sid,'label':int(label),'top1_prediction':int(choices[0]),
             'top5_predictions':choices.tolist(),'correct_top1':int(choices[0]==label),
             'correct_top5':int((choices==label).any())} for sid,label,choices in zip(sample_ids,labels.cpu(),top)]

def _probe_config(config):
    probe=copy.deepcopy(config.get('probe',config)); seeds=config.get('seeds',{})
    return {'epochs':int(probe.get('epochs',2)),'batch_size':int(probe.get('batch_size',256)),
            'lr':float(probe.get('lr',.01)),'weight_decay':float(probe.get('weight_decay',1e-4)),
            'standardize':bool(probe.get('standardize',True)),'num_classes':int(probe.get('num_classes',200)),
            'seed':int(probe.get('seed',seeds.get('probe',4404))),'order_seed':int(probe.get('order_seed',seeds.get('probe',4404)+1)),
            'selection':'best_dev_top1_then_earliest','device':str(probe.get('device','cpu'))}

def _probe_rng_state():
    numpy=np.random.get_state()
    return {'python':random.getstate(),'numpy':(numpy[0],numpy[1].tolist(),numpy[2],numpy[3],numpy[4]),
            'torch_cpu':torch.get_rng_state(),'torch_cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def _restore_probe_rng(state):
    random.setstate(state['python']); numpy=state['numpy']
    np.random.set_state((numpy[0],np.array(numpy[1],dtype=np.uint32),numpy[2],numpy[3],numpy[4]))
    torch.set_rng_state(state['torch_cpu'])
    if state['torch_cuda']: torch.cuda.set_rng_state_all(state['torch_cuda'])

def _save_probe_checkpoint(folder,saved,is_best=False):
    manifest_path=folder/'probe_manifest.json'
    old=json.loads(manifest_path.read_text()) if manifest_path.exists() else {'retained':[],'sequence':0,'best':None}
    sequence=old['sequence']+1
    path=folder/f'checkpoint-{sequence:06d}-update-{saved["successful_updates"]:07d}.pt'
    atomic_torch(path,saved)
    entry={'file':path.name,'sha256':digest_file(path),'completed_epochs':saved['completed_epochs'],
           'successful_updates':saved['successful_updates'],'sequence':sequence}
    best=entry if is_best else old.get('best')
    history=sorted(old['retained']+[entry],key=lambda x:x['sequence'])
    keep={row['file'] for row in history[-2:]}
    if best is not None: keep.add(best['file'])
    retained=[row for row in history if row['file'] in keep]
    manifest={'schema':PROBE_SCHEMA,'latest':entry,'retained':retained,'best':best,'sequence':sequence,
              'identity_hash':stable_digest(saved['identity']),
              'resume_function':'fmm_jepa.tiny_evaluation.fit_linear_probe with identical caches/config/model_id/output_dir',
              'resume_cell':'Notebook 01 frozen linear probe section','checkpoint_policy':'latest two valid resumable states and best development epoch'}
    atomic_json(manifest_path,manifest)
    for row in history:
        if row['file'] not in keep: (folder/row['file']).unlink(missing_ok=True)

def _load_probe_checkpoint(folder,identity):
    manifest_path=folder/'probe_manifest.json'
    if not manifest_path.exists():
        if list(folder.glob('checkpoint-*.pt')): raise RuntimeError('Orphan probe checkpoint without manifest; inspect before recovery')
        return None
    manifest=json.loads(manifest_path.read_text())
    if manifest.get('schema')!=PROBE_SCHEMA or manifest.get('identity_hash')!=stable_digest(identity): raise RuntimeError('Stale probe manifest identity')
    for entry in manifest['retained']:
        path=folder/entry['file']
        if not path.is_file() or digest_file(path)!=entry['sha256']: raise RuntimeError('Invalid retained probe checkpoint; never silently restart')
    saved=torch.load(folder/manifest['latest']['file'],map_location='cpu',weights_only=True)
    if saved.get('schema')!=PROBE_SCHEMA or saved.get('identity')!=identity: raise RuntimeError('Stale probe resume identity')
    return saved

def fit_linear_probe(train_cache, dev_cache, output_dir, config, model_id, encoder=None, stop_after_updates=None):
    """One writer per model. Authoritative output_dir belongs under checkpoints/<run_id>.
    Strictly Linear(D,200), fixed identical budgets and train-only standardization.
    Resume snapshots every epoch, 250 probe updates or 15 minutes; retain latest two+best.
    stop_after_updates is a controlled interruption hook for the resume certificate.
    """
    import fcntl
    folder=Path(output_dir)/model_id;folder.mkdir(parents=True,exist_ok=True)
    with (folder/'.writer.lock').open('a+') as lock:
        try: fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError as error: raise RuntimeError('Another probe writer owns this run/model') from error
        try:
            with frozen_encoder(encoder) if encoder is not None else nullcontext():
                return _fit_linear_probe(train_cache,dev_cache,folder,config,model_id,encoder,stop_after_updates)
        finally: fcntl.flock(lock,fcntl.LOCK_UN)

def _fit_linear_probe(train_cache,dev_cache,folder,config,model_id,encoder,stop_after_updates):
    for cache in (train_cache,dev_cache): validate_feature_cache(cache,cache['identity'])
    if set(train_cache['sample_ids']) & set(dev_cache['sample_ids']): raise ValueError('Probe train/dev overlap')
    for field in REQUIRED_IDENTITY:
        if field!='sample_ids' and train_cache['identity'][field]!=dev_cache['identity'][field]:
            raise ValueError('Train/dev representation identity mismatch: '+field)
    cfg=_probe_config(config)
    if cfg['num_classes']!=200 or cfg['epochs']<1 or cfg['batch_size']<1: raise ValueError('Invalid fixed probe configuration')
    identity={'config':cfg,'train_cache':train_cache['content_hash'],'dev_cache':dev_cache['content_hash'],
              'train_identity':train_cache['identity'],'dev_identity':dev_cache['identity'],'model_id':model_id}
    before=encoder_state_hash(encoder) if encoder is not None else None
    checkpoint_hash=train_cache['identity']['checkpoint_hash']
    if before is not None and before!=checkpoint_hash: raise ValueError('Probe encoder does not match cached teacher')
    device=torch.device(cfg['device']); stats=fit_standardization(train_cache['features'],cfg['standardize'])
    xtrain=standardize(train_cache['features'],stats); xdev=standardize(dev_cache['features'],stats)
    ytrain=train_cache['labels']; ydev=dev_cache['labels']; epochs=[]; times=[]; start_epoch=0; updates=0
    active_order=None;next_offset=0
    with preserve_randomness():
        torch.manual_seed(cfg['seed']); classifier=nn.Linear(xtrain.shape[1],200).to(device)
        initialization_hash=encoder_state_hash(classifier)
        optimizer=torch.optim.AdamW(classifier.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
        generator=torch.Generator().manual_seed(cfg['order_seed'])
        best={'top1':-1.0,'epoch':None,'state':None}; resumed=False
        saved=_load_probe_checkpoint(folder,identity)
        if saved is not None:
            classifier.load_state_dict(saved['classifier']); optimizer.load_state_dict(saved['optimizer'])
            generator.set_state(saved['order_rng']); epochs=saved['metrics']; start_epoch=saved['completed_epochs'];best=saved['best']
            updates=saved['successful_updates'];active_order=saved['active_order'];next_offset=saved['next_offset'];resumed=True
            _restore_probe_rng(saved['rng'])
        def snapshot(completed_epochs,order,offset,is_best=False):
            payload={'schema':PROBE_SCHEMA,'identity':identity,
                     'classifier':{k:v.detach().cpu().clone() for k,v in classifier.state_dict().items()},
                     'optimizer':optimizer.state_dict(),'order_rng':generator.get_state(),'rng':_probe_rng_state(),
                     'completed_epochs':completed_epochs,'successful_updates':updates,'attempted_updates':updates,
                     'active_order':order,'next_offset':offset,'metrics':epochs,'best':best,'standardization':stats,
                     'initialization_hash':initialization_hash,'scheduler':{'kind':'constant','lr':cfg['lr'],'successful_updates':updates},
                     'scaler':None,'precision':'float32','sampler_state':{'epoch':completed_epochs,'next_offset':offset},
                     'feature_cache_identity':identity,'resume_contract':'no prefetch; exact saved permutation+next consumed feature offset'}
            _save_probe_checkpoint(folder,payload,is_best=is_best)
        _sync(device); baseline=_memory_start(device); started=time.perf_counter();last_save=time.monotonic();start_updates=updates;new_exposures=0
        for epoch in range(start_epoch,cfg['epochs']):
            epoch_tick=time.perf_counter();classifier.train()
            order=active_order if active_order is not None else torch.randperm(len(xtrain),generator=generator)
            offset=next_offset if active_order is not None else 0
            for begin in range(offset,len(order),cfg['batch_size']):
                indices=order[begin:begin+cfg['batch_size']]
                _sync(device); tick=time.perf_counter(); optimizer.zero_grad(set_to_none=True)
                loss=F.cross_entropy(classifier(xtrain[indices].to(device)),ytrain[indices].to(device))
                if not torch.isfinite(loss): raise FloatingPointError('Nonfinite probe loss')
                loss.backward()
                if not all(p.grad is None or torch.isfinite(p.grad).all() for p in classifier.parameters()): raise FloatingPointError('Nonfinite probe gradient')
                optimizer.step();updates+=1;new_exposures+=len(indices)
                if not all(torch.isfinite(p).all() for p in classifier.parameters()): raise FloatingPointError('Nonfinite probe parameters')
                _sync(device);times.append(time.perf_counter()-tick)
                consumed=begin+len(indices)
                interrupt=stop_after_updates is not None and updates>=stop_after_updates
                if updates%250==0 or time.monotonic()-last_save>=900 or interrupt:
                    snapshot(epoch,order,consumed);last_save=time.monotonic()
                if interrupt:
                    return {'status':'INTERRUPTED','successful_updates':updates,'completed_epochs':epoch,
                            'resume_manifest':str(folder/'probe_manifest.json')}
            active_order=None;next_offset=0;classifier.eval()
            with torch.inference_mode():
                train_logits=torch.cat([classifier(part.to(device)).cpu() for part in xtrain.split(cfg['batch_size'])])
                dev_logits=torch.cat([classifier(part.to(device)).cpu() for part in xdev.split(cfg['batch_size'])])
            train=classification_metrics(train_logits,ytrain);dev=classification_metrics(dev_logits,ydev)
            for split,metrics in (('train',train),('development',dev)):
                epochs.append({'model_id':model_id,'seed':cfg['seed'],'phase':'linear_probe','split':split,
                               'checkpoint_hash':checkpoint_hash,'epoch':epoch+1,'epoch_wall_seconds':time.perf_counter()-epoch_tick,
                               'permutation_hash':encoder_state_hash({'order':order}),**metrics})
            improved=dev['top1']>best['top1']
            if improved: best={'top1':dev['top1'],'epoch':epoch+1,'state':{k:v.detach().cpu().clone() for k,v in classifier.state_dict().items()}}
            snapshot(epoch+1,None,0,is_best=improved);last_save=time.monotonic();write_csv(folder/'epoch_metrics.csv',epochs)
        _sync(device);seconds=time.perf_counter()-started
        performance=_performance('probe_fitting',seconds,new_exposures,times,device,baseline,
                                 includes_data_loading=False,includes_preprocessing=False,includes_diagnostics=False,
                                 includes_optimizer=True,includes_epoch_evaluation=True,includes_checkpoint_writes=True,
                                 batch_size=cfg['batch_size'],completed_epochs=cfg['epochs'],new_epochs=cfg['epochs']-start_epoch,
                                 successful_probe_updates=updates,new_probe_updates=updates-start_updates)
        classifier.load_state_dict(best['state']);classifier.eval();probe_hash=encoder_state_hash(classifier)
        with torch.inference_mode(): logits=torch.cat([classifier(part.to(device)).cpu() for part in xdev.split(cfg['batch_size'])])
        predictions=prediction_rows(logits,ydev,dev_cache['sample_ids'],model_id,'development',checkpoint_hash,probe_hash,cfg['seed'])
        result=classification_metrics(logits,ydev)
        selected={'schema':PROBE_SCHEMA,'state_dict':best['state'],'standardization':stats,'config':cfg,
                  'encoder_checkpoint_hash':checkpoint_hash,'probe_hash':probe_hash,'selected_epoch':best['epoch'],
                  'choices_fixed':True,'input_dim':xtrain.shape[1],'identity':identity}
        atomic_torch(folder/'selected_probe.pt',selected);write_csv(folder/'development_predictions.csv',predictions)
    after=encoder_state_hash(encoder) if encoder is not None else None
    if before!=after: raise RuntimeError('Probe fitting mutated encoder')
    summary={'model_id':model_id,'seed':cfg['seed'],'phase':'linear_probe','split':'development','checkpoint_hash':checkpoint_hash,
             'probe_hash':probe_hash,'selected_epoch':best['epoch'],'completed_epochs':cfg['epochs'],'successful_probe_updates':updates,'budget':cfg,
             'initialization_hash':initialization_hash,'encoder_unchanged':before==after if encoder is not None else None,
             'encoder_verification':'state_hash_before_after' if encoder is not None else 'NOT_RUN_no_encoder_passed',
             'train_examples':len(xtrain),'train_classes':int(ytrain.unique().numel()),'initialization_control':'untrained' in model_id,
             'standardization_fit_scope':'probe_train_only','resumed':resumed,'metrics':result,
             'selected_probe_path':str(folder/'selected_probe.pt'),'predictions_path':str(folder/'development_predictions.csv'),
             'resume_manifest':str(folder/'probe_manifest.json')}
    atomic_json(folder/'summary.json',summary)
    return {'summary':summary,'metrics':epochs,'predictions':predictions,'performance':performance,'selected_probe':selected}

def evaluate_heldout(heldout_cache, selected_probe, config, model_id, output_dir):
    if config.get('run_mode')!='pilot' or not config.get('evaluation',{}).get('heldout_enabled',False):
        raise ValueError('Held-out evaluation requires explicit pilot mode and heldout_enabled=true')
    if not selected_probe.get('choices_fixed'): raise ValueError('Freeze all checkpoint/probe choices before held-out evaluation')
    validate_feature_cache(heldout_cache,heldout_cache['identity'])
    chosen=selected_probe['identity']
    used=set(chosen['train_identity']['sample_ids']) | set(chosen['dev_identity']['sample_ids'])
    if used & set(heldout_cache['sample_ids']): raise ValueError('Held-out overlaps training/development')
    for field in REQUIRED_IDENTITY:
        if field!='sample_ids' and heldout_cache['identity'][field]!=chosen['train_identity'][field]: raise ValueError('Held-out representation mismatch')
    with preserve_randomness(),torch.inference_mode():
        classifier=nn.Linear(selected_probe['input_dim'],200)
        classifier.load_state_dict(selected_probe['state_dict']); classifier.eval()
        features=standardize(heldout_cache['features'],selected_probe['standardization'])
        logits=classifier(features)
        metrics=classification_metrics(logits,heldout_cache['labels'])
        predictions=prediction_rows(logits,heldout_cache['labels'],heldout_cache['sample_ids'],model_id,'heldout',
                                    heldout_cache['identity']['checkpoint_hash'],selected_probe['probe_hash'],selected_probe['config']['seed'])
    write_csv(Path(output_dir)/(model_id+'_heldout_predictions.csv'),predictions)
    return {'metrics':metrics,'predictions':predictions}

def embedding_diagnostics(features,tokens=None,variance_epsilon=1e-12,low_rank_threshold=2.0):
    """Dispersion BEFORE standardization; matching-position token std is across images."""
    features=features.detach().double().cpu()
    if features.ndim!=2 or len(features)<2 or not torch.isfinite(features).all(): raise ValueError('Diagnostics require >=2 finite image embeddings')
    centered=features-features.mean(0); eigenvalues=torch.linalg.eigvalsh(centered.T @ centered/(len(features)-1)).clamp_min(0)
    total=float(eigenvalues.sum())
    positive=eigenvalues[eigenvalues>0]
    rank=float(torch.exp(-((positive/total)*(positive/total).log()).sum())) if total>0 else 0.0
    norms=features.norm(dim=1); normalized=F.normalize(features,dim=1,eps=1e-12)
    cosines=(normalized@normalized.T)[torch.triu(torch.ones(len(features),len(features),dtype=torch.bool),diagonal=1)]
    rank_ceiling=min(features.shape[1],len(features)-1)
    result={'diagnostic_examples':len(features),'feature_dim':features.shape[1],
            'feature_std_mean':float(features.std(0,unbiased=False).mean()),
            'feature_std_min':float(features.std(0,unbiased=False).min()),
            'feature_std_max':float(features.std(0,unbiased=False).max()),
            'embedding_norm_mean':float(norms.mean()),'embedding_norm_min':float(norms.min()),
            'between_image_cosine_mean':float(cosines.mean()),'between_image_cosine_std':float(cosines.std(unbiased=False)),
            'zero_norm_examples':int((norms<=1e-12).sum()),'covariance_trace':total,'effective_rank':min(rank,float(rank_ceiling)),
            'effective_rank_ceiling':rank_ceiling,'covariance_eigenvalues':eigenvalues.tolist(),
            'zero_variance_flag':total<=variance_epsilon,'very_low_rank_flag':rank<low_rank_threshold,
            'variance_epsilon':variance_epsilon,'low_rank_threshold':low_rank_threshold,
            'threshold_interpretation':'registered diagnostic flags; advisory, not universal collapse theorems',
            'dispersion_stage':'before_optional_train_fitted_standardization'}
    if tokens is not None:
        tokens=tokens.detach().float().cpu()
        if tokens.ndim!=3 or len(tokens)!=len(features): raise ValueError('Expected tokens [images,positions,features]')
        result['matching_patch_across_images_std_mean']=float(tokens.std(dim=0,unbiased=False).mean())
        result['patch_positions']=tokens.shape[1]
    return result

def compute_diagnostics(encoder, dataset_or_images, config=None):
    config=config or {}; evaluation=config.get('evaluation',config)
    batch_size=int(evaluation.get('diagnostic_batch_size',evaluation.get('batch_size',32)))
    device=next(encoder.parameters()).device
    tokens=[]; ids=[]; branch=[]
    with frozen_encoder(encoder),torch.inference_mode():
        for start in range(0,len(dataset_or_images),batch_size):
            if isinstance(dataset_or_images,torch.Tensor): images=dataset_or_images[start:start+batch_size]
            else:
                images,_,batch_ids=_batch(dataset_or_images,start,min(start+batch_size,len(dataset_or_images)));ids.extend(batch_ids)
            output=encoder(images.to(device,dtype=torch.float32)); tokens.append(output.float().cpu().clone())
            for block in getattr(encoder,'blocks',[]):
                stats=getattr(block.mixer,'last_metrics',{})
                if stats: branch.append({key:float(stats[key]) for key in ('near_norm','far_norm') if key in stats})
    tokens=torch.cat(tokens)
    result=embedding_diagnostics(tokens.mean(1),tokens,
                variance_epsilon=float(config.get('diagnostics',{}).get('zero_variance_epsilon',evaluation.get('variance_epsilon',1e-12))),
                low_rank_threshold=float(config.get('diagnostics',{}).get('low_effective_rank_warning',evaluation.get('low_rank_threshold',2.0))))
    result['checkpoint_hash']=encoder_state_hash(encoder)
    result['sample_ids_hash']=stable_digest(ids) if ids else None
    result['encoder_kind']=getattr(encoder,'kind','unknown')
    result['encoder_choice']='ema_target'; result['pooling']='mean_final_layernorm_patches'
    for key in ('near_norm','far_norm'):
        values=[row[key] for row in branch if key in row]
        result['fullimage_'+key]=sum(values)/len(values) if values else None
    result['branch_interpretation']='Nonzero output establishes activity, not semantic usefulness; training gradient norms reported separately.'
    return result

def paired_bootstrap(first,second,repeats=1000,seed=5505,metric='correct_top1'):
    a={row['sample_id']:row for row in first}; b={row['sample_id']:row for row in second}
    if len(a)!=len(first) or len(b)!=len(second) or set(a)!=set(b) or not a: raise ValueError('Paired predictions require unique identical sample memberships')
    ids=sorted(a)
    if any(a[sid]['label']!=b[sid]['label'] for sid in ids): raise ValueError('Paired predictions disagree on true labels')
    if any(a[sid]['split']!=b[sid]['split'] for sid in ids): raise ValueError('Paired predictions mix splits')
    differences=np.array([a[sid][metric]-b[sid][metric] for sid in ids],dtype=np.float64)
    rng=np.random.default_rng(seed); draws=[]
    for _ in range(int(repeats)): draws.append(float(differences[rng.integers(0,len(ids),len(ids))].mean()))
    low,high=np.percentile(draws,[2.5,97.5])
    return {'delta_percentage_points':float(differences.mean()*100),'ci95_percentage_points':[float(low*100),float(high*100)],
            'examples':len(ids),'bootstrap_repeats':int(repeats),'bootstrap_seed':int(seed),'metric':metric,
            'uncertainty_scope':'paired evaluation-image bootstrap conditional on these fixed trained models; excludes training-seed variability'}

def compare_representations(results,config):
    required=('vit_untrained','vit_trained','fmm_untrained','fmm_trained')
    if set(required)-set(results): raise ValueError('All four initialization/trained controls are required')
    budgets=[results[key]['summary']['budget'] for key in required]
    if any(budget!=budgets[0] for budget in budgets[1:]): raise ValueError('Unequal probe budgets')
    initializations=[results[key]['summary']['initialization_hash'] for key in required]
    if len(set(initializations))!=1: raise ValueError('Probe initialization must match for same-width representations')
    repeats=config.get('probe',{}).get('bootstrap_repeats',1000);seed=config.get('seeds',{}).get('bootstrap',5505)
    pairs={'delta_vit':('vit_trained','vit_untrained'),'delta_fmm':('fmm_trained','fmm_untrained'),
           'delta_architecture':('fmm_trained','vit_trained')}
    deltas={name:{metric:paired_bootstrap(results[a]['predictions'],results[b]['predictions'],repeats,seed,metric)
                 for metric in ('correct_top1','correct_top5')} for name,(a,b) in pairs.items()}
    smoke=config.get('run_mode','smoke')=='smoke'
    return {'schema':'fmm-jepa-tiny-comparison/v1','run_mode':config.get('run_mode','smoke'),
            'results':{key:results[key]['summary'] for key in required},'deltas':deltas,
            'learning_evidence':'inconclusive' if smoke else 'requires_completed_budget_and_diagnostic_review',
            'ready_for_notebook_02':False,'training_seeds':1,'heldout_status':'NOT_RUN',
            'interpretation':'Integration-only short smoke; learning is unestablished regardless of score.' if smoke else
                             'One-seed pilot. Interpret trained-minus-initial deltas with budget and collapse evidence; no robust architectural conclusion.',
            'chance_reference_top1':.005,'chance_role':'sanity reference only; initialization controls are the learning baseline'}
